In [ ]:
# Cell 1: 라이브러리 설치
!pip install pandas tqdm openai
!pip install ipywidgets
!pip install mlxtend


In [ ]:
# Cell 2 (수정본): CSV 병합 + 정제
import os
import glob
import pandas as pd

DATA_DIR   = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\1661 - 1722"
OUTPUT_DIR = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_CSV = os.path.join(OUTPUT_DIR, "merged.csv")

# ───── 사전 제거할 명백한 허사 (1글자) ─────
# NER API 비용 절감용. 어차피 NONE으로 분류될 게 확실한 한자.
STOP_CHARS = set([
    "之", "也", "以", "而", "於", "其", "者", "焉", "乎", "矣",
    "曰", "云", "等", "及", "與", "或", "亦", "又", "則", "且",
    "於", "為", "為", "以", "於", "之", "於",
    "是", "此", "彼", "斯", "茲",
    "上", "下", "中", "內", "外",
    "有", "無", "不", "未", "非", "勿", "毋",
    "可", "能", "得", "當", "宜", "應",
    "皆", "俱", "悉", "盡", "並", "共",
    "甚", "至", "極", "最",
    "已", "既", "嘗", "曾",
    "一", "二", "三", "四", "五", "六", "七", "八", "九", "十",
    "百", "千", "萬",
])

all_dfs = []
csv_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))
print(f"[INFO] 발견된 파일 수: {len(csv_files)}개")

skipped_files = 0
for filepath in csv_files:
    filename = os.path.basename(filepath)
    name     = filename.replace(".csv", "")

    try:
        parts = name.split("_")
        year = int(parts[0])

        if parts[1].startswith("yun"):
            is_yun = True
            month  = int(parts[1].replace("yun", ""))
        else:
            is_yun = False
            month  = int(parts[1])
    except (ValueError, IndexError):
        print(f"[WARN] 파일명 형식 불일치, 건너뜀: {filename}")
        skipped_files += 1
        continue

    try:
        df = pd.read_csv(filepath, encoding="utf-8-sig")
    except UnicodeDecodeError:
        df = pd.read_csv(filepath, encoding="cp949")

    df.columns = ["word", "freq"]

    # ───── 정제 ─────
    # 1. word NaN 제거
    df = df.dropna(subset=["word"])
    # 2. word를 문자열로 강제 + 공백 제거
    df["word"] = df["word"].astype(str).str.strip()
    # 3. 빈 문자열 제거
    df = df[df["word"] != ""]
    # 4. freq 숫자 변환 + 양수만
    df["freq"] = pd.to_numeric(df["freq"], errors="coerce")
    df = df.dropna(subset=["freq"])
    df = df[df["freq"] > 0]
    # 5. freq를 정수로
    df["freq"] = df["freq"].astype(int)
    # 6. 1글자 허사 제거
    df = df[~df["word"].isin(STOP_CHARS)]

    # 메타 컬럼
    df["year"]        = year
    df["month"]       = month
    df["is_yun"]      = is_yun
    df["period_year"] = str(year)
    if is_yun:
        df["period_month"] = f"{year:04d}-yun{month:02d}"
    else:
        df["period_month"] = f"{year:04d}-{month:02d}"

    all_dfs.append(df)

merged = pd.concat(all_dfs, ignore_index=True)

# ───── 같은 (period_month, word) 중복 시 freq 합산 ─────
before = len(merged)
merged = (
    merged.groupby(
        ["year", "month", "is_yun", "period_year", "period_month", "word"],
        as_index=False
    )["freq"].sum()
)
after = len(merged)
if before != after:
    print(f"[INFO] 중복 합산: {before:,} → {after:,}행")

merged = merged[[
    "year", "month", "is_yun",
    "period_year", "period_month",
    "word", "freq"
]].sort_values(["year", "month", "is_yun"]).reset_index(drop=True)

merged.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

# ───── 통계 ─────
print(f"\n[DONE] 병합 완료 → {OUTPUT_CSV}")
print(f"  처리된 파일 : {len(csv_files) - skipped_files:,}개 (건너뜀 {skipped_files}개)")
print(f"  총 행 수    : {len(merged):,}행")
print(f"  기간        : {merged['period_month'].min()} ~ {merged['period_month'].max()}")
print(f"  고유 단어   : {merged['word'].nunique():,}개")
print(f"  일반 달     : {merged[~merged['is_yun']]['period_month'].nunique():,}개월")
print(f"  윤달        : {merged[merged['is_yun']]['period_month'].nunique():,}개월")

# 단어 길이 분포
merged["wlen"] = merged["word"].str.len()
print(f"\n[단어 길이 분포 (고유 단어 기준)]")
unique_words = merged.drop_duplicates("word")
print(unique_words["wlen"].value_counts().sort_index().to_string())

print(f"\n[샘플]")
print(merged.drop(columns=["wlen"]).head(10).to_string(index=False))


In [2]:
# Cell 3 (수정본): NER 설정
import os
import re
import json
import time
import pandas as pd
from tqdm import tqdm
from openai import OpenAI
from dotenv import load_dotenv 

INPUT_CSV      = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\merged.csv"
OUTPUT_ALL_CSV = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\ner_results_all.csv"
OUTPUT_CSV     = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\ner_results.csv"
CHECKPOINT_CSV = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\ner_checkpoint.csv"
load_dotenv(r"C:\Users\USER\OneDrive\바탕 화면\nerproject\.env")

# ───── API 설정 ─────
DEEPSEEK_API_KEY  = os.environ.get("DEEPSEEK_API_KEY", "")
DEEPSEEK_BASE_URL = "https://api.deepseek.com"
MODEL_NAME        = "deepseek-v4-flash"  # ← deepseek-v4-pro로 변경도 고려 (역사적 맥락을 이해하지 못하는 경우, 다하고 지워라 빨리)

# ───── 실행 파라미터 ─────
BATCH_SIZE   = 30 # 혹은 50
DELAY_API    = 1.5
CHECKPOINT_N = 10

# ───── 검증 ─────
if DEEPSEEK_API_KEY:
    masked = DEEPSEEK_API_KEY[:7] + "..." + DEEPSEEK_API_KEY[-4:]
    print(f"[OK] API 키 로드 완료: {masked}")
else:
    print("[ERROR] API 키를 .env에서 못 찾음. .env 파일 위치와 변수명 확인하세요.")

client = OpenAI(
    api_key=DEEPSEEK_API_KEY or "PLACEHOLDER",
    base_url=DEEPSEEK_BASE_URL
)

SYSTEM_PROMPT = """너는 청나라 실록(淸實錄) 한문 단어를 분류하는 전문가다.
주어진 한문 단어 목록 각각을 아래 카테고리 중 하나로 분류하라.

【카테고리】
- PER  : 인물 (고유 인명. 황제 본명, 신하, 장군 등)
- LOC  : 지명 (지역, 도시, 궁궐, 강, 산 등)
- OFF  : 관직/직책/작위 (大學士, 總督, 巡撫, 貝勒, 公 등)
- GRP  : 민족/집단/세력/부족 (蒙古, 準噶爾, 三藩, 科爾沁 등)
- DATE : 연호/날짜/간지 (康熙, 正月, 辛亥, 甲子 등)
- SYS  : 제도/법령/정책 (科擧, 海禁, 八旗 등)
- EVT  : 사건/전쟁/반란 (三藩之亂 등)
- NONE : 위 카테고리에 해당 없음 (일반어, 동사, 조사, 모호한 표현 등)

【분류 예시 — 정상 케이스】
- "索尼"        → PER
- "鰲拜"        → PER
- "遏必隆"      → PER
- "麻勒吉"      → PER
- "大學士索尼"  → PER  (관직+인명 결합형은 PER)
- "總督于成龍"  → PER  (관직+인명 결합형은 PER)
- "大學士"      → OFF  (관직 단독)
- "巡撫"        → OFF
- "侍郎"        → OFF
- "貝勒"        → OFF  (작위)
- "公"          → OFF  (작위)
- "盛京"        → LOC
- "乾清門"      → LOC  (궁궐)
- "黃河"        → LOC
- "蒙古"        → GRP
- "準噶爾"      → GRP
- "科爾沁"      → GRP  (몽골 부족)
- "三藩"        → GRP
- "康熙"        → DATE
- "正月"        → DATE
- "辛亥"        → DATE  (간지)
- "科擧"        → SYS
- "海禁"        → SYS
- "三藩之亂"    → EVT

【분류 예시 — 주의해야 할 오류 케이스】
- "大行"        → NONE  ('大行皇帝(승하한 황제)' 일부, 고유명사 아님)
- "今上"        → NONE  ('현 황제' 일반 지칭, 고유명사 아님)
- "皇上"        → NONE  (일반 호칭)
- "上"          → NONE  (단독으로는 모호)
- "帝"          → NONE  (직위 일반어)
- "世祖"        → DATE  (묘호. 특정 황제 지칭이지만 고유명사성 약함 — 묘호는 DATE로)
- "王貝勒"      → OFF   (작위 나열, 인명 아님)
- "王公"        → OFF   (작위 나열)
- "四子"        → NONE  ('넷째 아들' 일반어. '四子部落' 같은 부족명 아니면 NONE)
- "宗室"        → GRP   (황실 종친 집단)
- "滿漢"        → GRP   (만주족+한족)
- "蘇克薩哈遏"  → NONE  (두 인명 '蘇克薩哈'+'遏必隆'이 잘못 붙음 → NONE)
- "任學士"      → NONE  ('任'이 동사 '임명하다'일 가능성 → NONE)
- "自慎以"      → NONE  (토크나이저 오류 조각)
- "之後萬"      → NONE  (토크나이저 오류 조각)
- "海面"        → NONE  (일반명사)
- "流民"        → NONE  (일반명사)
- "百姓"        → NONE  (일반명사)
- "朝廷"        → NONE  (일반명사)

【분류 규칙】
1. 반드시 JSON만 반환. 설명 텍스트 절대 없음.
2. 형식: {"단어": "카테고리", "단어": "카테고리", ...}
3. 입력된 단어를 빠짐없이 전부 분류.
4. 카테고리 값은 반드시 PER/LOC/OFF/GRP/DATE/SYS/EVT/NONE 중 하나.
5. 관직+인명 결합형(예: '大學士某某')은 PER.
6. 두 인명이 잘못 붙은 것 같은 단어(어색한 음역 결합)는 NONE.
7. 황제 일반 지칭(大行/今上/皇上/帝/上)은 NONE.
8. 묘호(世祖/聖祖/太祖)는 DATE.
9. 일반명사(百姓/朝廷/流民/海面)는 NONE.
10. 토크나이저 오류로 보이는 의미 불명 조각은 NONE.
11. 확신이 없으면 NONE.
"""

print("[INFO] 설정 완료")
print(f"  - 모델       : {MODEL_NAME}")
print(f"  - 배치 크기  : {BATCH_SIZE}")
print(f"  - API 대기   : {DELAY_API}초")
print(f"  - 체크포인트 : {CHECKPOINT_N}배치마다")
if not DEEPSEEK_API_KEY:
    print("[WARN] API 키가 비어있습니다. 실행 전에 채워 넣으세요.")


[OK] API 키 로드 완료: sk-2498...b45c
[INFO] 설정 완료
  - 모델       : deepseek-v4-flash
  - 배치 크기  : 30
  - API 대기   : 1.5초
  - 체크포인트 : 10배치마다


In [3]:
# Cell 4 (수정본): NER 함수 정의
VALID_CATEGORIES = {"PER", "LOC", "OFF", "GRP", "DATE", "SYS", "EVT", "NONE"}


def call_ner(words: list, retries: int = 2) -> dict:
    """
    한문 단어 리스트를 카테고리로 분류.
    - 성공: {word: category}
    - API 오류 / JSON 파싱 실패: 해당 단어들을 "UNK"로 반환 (다음 실행 시 재시도)
    - 모델이 일부 단어를 누락하면 누락분만 자동 재요청
    """
    if not DEEPSEEK_API_KEY:
        return {w: "NONE" for w in words}

    word_list = "\n".join(f"- {w}" for w in words)

    for attempt in range(retries + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": f"다음 단어들을 분류하라:\n{word_list}"}
                ],
                temperature=0.0,
                max_tokens=8192, # 혹은 4096
                response_format={"type": "json_object"},  # JSON 강제 (호환 안 되면 이 줄 삭제)
            )
            raw = resp.choices[0].message.content.strip()
            raw = re.sub(r"```json\s*", "", raw)
            raw = re.sub(r"```\s*", "", raw)
            parsed = json.loads(raw)

            # 카테고리 화이트리스트 검증 (PERSON, PLACE 같은 변형 → NONE으로 강제)
            for w, c in list(parsed.items()):
                if c not in VALID_CATEGORIES:
                    parsed[w] = "NONE"

            # 누락 단어 검출 → 누락분만 재요청
            missing = [w for w in words if w not in parsed]
            if missing:
                if attempt < retries:
                    print(f"[RETRY] 누락 {len(missing)}개 재요청")
                    sub = call_ner(missing, retries=0)
                    parsed.update(sub)
                    for w in words:
                        if w not in parsed:
                            parsed[w] = "UNK"
                else:
                    for w in missing:
                        parsed[w] = "UNK"

            return parsed

        except json.JSONDecodeError as e:
            print(f"[WARN] JSON 파싱 실패 (시도 {attempt+1}/{retries+1}): {e}")
            if attempt == retries:
                return {w: "UNK" for w in words}
            time.sleep(2)
        except Exception as e:
            print(f"[WARN] API 오류 (시도 {attempt+1}/{retries+1}): {e}")
            if attempt == retries:
                return {w: "UNK" for w in words}
            time.sleep(3)

    return {w: "UNK" for w in words}


print("[INFO] 함수 정의 완료")


[INFO] 함수 정의 완료


In [4]:
# Cell 5 (수정본): NER 실행 — UNK 자동 재시도 + 안전 저장
df = pd.read_csv(INPUT_CSV)
all_words = df["word"].dropna().astype(str).unique().tolist()
print(f"[INFO] 전체 고유 단어: {len(all_words):,}개")

# 체크포인트 로드
done_map = {}
if os.path.exists(CHECKPOINT_CSV):
    ck = pd.read_csv(CHECKPOINT_CSV)
    done_map = dict(zip(ck["word"].astype(str), ck["category"].astype(str)))
    n_unk = sum(1 for v in done_map.values() if v == "UNK")
    print(f"[INFO] 체크포인트 로드: {len(done_map):,}개 (UNK {n_unk:,}개는 재시도)")

# 처리 대상: 미처리 + UNK
todo_words = [
    w for w in all_words
    if (w not in done_map) or (done_map.get(w) == "UNK")
]
print(f"[INFO] 처리 대상: {len(todo_words):,}개")

batches = [todo_words[i:i+BATCH_SIZE] for i in range(0, len(todo_words), BATCH_SIZE)]
print(f"[INFO] 배치 수: {len(batches)}개\n")


def save_checkpoint(done_map):
    ck_df = pd.DataFrame(list(done_map.items()), columns=["word", "category"])
    ck_df.to_csv(CHECKPOINT_CSV, index=False, encoding="utf-8-sig")
    return ck_df


# ───── 메인 루프 (Ctrl+C 안전) ─────
try:
    for batch_idx, batch in enumerate(tqdm(batches, desc="NER 배치")):
        result = call_ner(batch)
        done_map.update(result)
        time.sleep(DELAY_API)

        if (batch_idx + 1) % CHECKPOINT_N == 0:
            ck_df = save_checkpoint(done_map)
            dist  = ck_df["category"].value_counts().to_dict()
            print(f"[CHECKPOINT] {len(done_map):,}개 저장 / 분포: {dist}")

except KeyboardInterrupt:
    print("\n[INTERRUPT] 사용자 중단 — 진행분 저장 중...")
finally:
    save_checkpoint(done_map)
    print(f"[SAFE] 체크포인트 저장 완료: {len(done_map):,}개")


# ───── 최종 결과 생성 ─────
ner_df = pd.DataFrame(list(done_map.items()), columns=["word", "category"])

n_unk = (ner_df["category"] == "UNK").sum()
if n_unk > 0:
    print(f"\n[WARN] UNK {n_unk:,}개 남음 — Cell 5를 다시 실행하면 자동 재시도합니다.")

# merged에 category 병합
df_out = df.merge(ner_df, on="word", how="left")
df_out["category"] = df_out["category"].fillna("NONE")

# 분석용: NONE / UNK 모두 제외
df_filtered = df_out[~df_out["category"].isin(["NONE", "UNK"])].reset_index(drop=True)

df_out.to_csv(OUTPUT_ALL_CSV, index=False, encoding="utf-8-sig")
df_filtered.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(f"\n[DONE] NER 완료")
print(f"  전체 (NONE/UNK 포함) → ner_results_all.csv  ({len(df_out):,}행)")
print(f"  분석용 (NONE/UNK 제외) → ner_results.csv      ({len(df_filtered):,}행)")
print(f"\n[카테고리 분포 — 단어 기준 (고유 단어)]")
print(ner_df['category'].value_counts().to_string())
print(f"\n[카테고리 분포 — 행 기준 (분석용 데이터)]")
print(df_filtered['category'].value_counts().to_string())


[INFO] 전체 고유 단어: 35,100개
[INFO] 체크포인트 로드: 26,461개 (UNK 0개는 재시도)
[INFO] 처리 대상: 8,693개
[INFO] 배치 수: 290개



NER 배치:   3%|▎         | 10/290 [03:22<1:43:58, 22.28s/it]

[CHECKPOINT] 26,761개 저장 / 분포: {'NONE': 22922, 'PER': 2012, 'LOC': 891, 'OFF': 436, 'SYS': 175, 'DATE': 173, 'GRP': 146, 'EVT': 6}


NER 배치:   7%|▋         | 20/290 [06:24<1:16:06, 16.91s/it]

[CHECKPOINT] 27,061개 저장 / 분포: {'NONE': 23152, 'PER': 2056, 'LOC': 909, 'OFF': 437, 'DATE': 180, 'SYS': 175, 'GRP': 146, 'EVT': 6}


...

NER 배치:  97%|█████████▋| 280/290 [1:27:32<03:14, 19.49s/it]

[CHECKPOINT] 34,861개 저장 / 분포: {'NONE': 29977, 'PER': 2668, 'LOC': 1123, 'OFF': 484, 'SYS': 221, 'DATE': 196, 'GRP': 183, 'EVT': 9}


NER 배치: 100%|██████████| 290/290 [1:30:10<00:00, 18.66s/it]

[CHECKPOINT] 35,154개 저장 / 분포: {'NONE': 30232, 'PER': 2688, 'LOC': 1131, 'OFF': 486, 'SYS': 221, 'DATE': 200, 'GRP': 186, 'EVT': 10}
[SAFE] 체크포인트 저장 완료: 35,154개



[DONE] NER 완료
  전체 (NONE/UNK 포함) → ner_results_all.csv  (102,861행)
  분석용 (NONE/UNK 제외) → ner_results.csv      (21,126행)

[카테고리 분포 — 단어 기준 (고유 단어)]
category
NONE    30232
PER      2688
LOC      1131
OFF       486
SYS       221
DATE      200
GRP       186
EVT        10

[카테고리 분포 — 행 기준 (분석용 데이터)]
category
OFF     5547
DATE    4761
PER     4406
LOC     4071
SYS     1173
GRP     1139
EVT       29


In [5]:
# Cell 5.1: NER 결과 진단
import pandas as pd

ck = pd.read_csv(CHECKPOINT_CSV)
merged = pd.read_csv(INPUT_CSV)

# ───── 1. 전체 분포 ─────
print("=" * 60)
print("[1] 전체 카테고리 분포")
print("=" * 60)
print(ck['category'].value_counts().to_string())
print(f"\nUNK 잔존: {(ck['category']=='UNK').sum()}개")
print(f"전체 단어: {len(ck):,}개")

# ───── 2. 카테고리별 샘플 (랜덤) ─────
print("\n" + "=" * 60)
print("[2] 카테고리별 랜덤 샘플 20개")
print("=" * 60)
for cat in ['PER','LOC','OFF','GRP','DATE','SYS','EVT']:
    n = (ck['category']==cat).sum()
    if n == 0:
        continue
    sample = ck[ck['category']==cat]['word'].sample(min(20, n), random_state=42).tolist()
    print(f"\n{cat} ({n}개):")
    print(f"  {sample}")

# ───── 3. NONE 품질 검증 ─────
print("\n" + "=" * 60)
print("[3] NONE 고빈도 TOP 50 — 진짜 NONE인지 검증")
print("=" * 60)
none_words = ck[ck['category']=='NONE']['word'].tolist()
none_freq = (merged[merged['word'].isin(none_words)]
             .groupby('word')['freq'].sum()
             .sort_values(ascending=False)
             .head(50))
print(none_freq.to_string())
print("\n→ 위 목록에 명백한 고유명사(인명/지명/관직)가 있는지 확인")

# ───── 4. 길이별 분포 ─────
print("\n" + "=" * 60)
print("[4] 글자수별 카테고리 분포 (비율)")
print("=" * 60)
ck['len'] = ck['word'].str.len()
print(pd.crosstab(ck['len'], ck['category'], normalize='index').round(2).to_string())
print("\n→ 1글자: NONE 많은 게 정상")
print("→ 3글자+: NONE 50% 미만이면 정상 (대부분 고유명사여야)")

# ───── 5. 지난번 문제 단어 재분류 확인 ─────
print("\n" + "=" * 60)
print("[5] 주요 검증 단어 분류 결과")
print("=" * 60)
check_words = ['大行', '今上', '皇上', '帝', '王貝勒', '王公', '四子',
               '世祖', '聖祖', '宗室', '滿漢', '康熙', '北京', '索尼', '鰲拜']
for w in check_words:
    cat = ck[ck['word']==w]['category'].values
    if len(cat) > 0:
        print(f"  {w:8s} → {cat[0]}")
    else:
        print(f"  {w:8s} → (단어 자체가 데이터에 없음)")

# ───── 6. 분석용 데이터 통계 (NONE/UNK 제외) ─────
print("\n" + "=" * 60)
print("[6] 분석용 데이터 (NONE/UNK 제외)")
print("=" * 60)
analyzable = ck[~ck['category'].isin(['NONE','UNK'])]
print(f"분석 대상 단어 수: {len(analyzable):,}개")
print(f"\n카테고리별:")
print(analyzable['category'].value_counts().to_string())


[1] 전체 카테고리 분포
category
NONE    30232
PER      2688
LOC      1131
OFF       486
SYS       221
DATE      200
GRP       186
EVT        10

UNK 잔존: 0개
전체 단어: 35,154개

[2] 카테고리별 랜덤 샘플 20개

PER (2688개):
  ['達爾瑪', '王毓賢', '圖爾格', '查祿', '蔣國正', '喇木扎', '學士馬', '吳昺', '碩端敏', '王善', '伯噶', '劉漢業', '王盛', '王化致', '郭多', '董宜斯噶卜', '屈盡美', '巡撫布雅', '德赫勒', '岳鍾琪']

LOC (1131개):
  ['保安', '新安', '饒南', '大宛', '烏闌布通', '歷城縣', '尤溪', '廣西', '武陟', '淮', '關外', '金山', '建業', '薩哈連', '大都', '青縣', '西暖閣', '湖州', '黑山', '洛河']

OFF (486개):
  ['司同知', '經略', '平郡王', '河員', '各扎薩克', '汗王', '漢軍學士', '護軍參領', '平西王', '鎮國將', '刑部', '蘇松常道', '左春坊', '陵遣官', '諸佐領', '故大學士', '運同', '巡撫', '今之督撫', '外任']

GRP (186개):
  ['六旗', '西番', '奈曼喀爾', '喀爾喀人', '旗漢', '左翼', '準噶爾賊眾', '左營', '郭爾羅斯扎魯特', '扎魯特喀爾', '提標', '正黃旗漢', '旗丁', '秦喀爾', '倭', '明朝', '科爾沁喀爾', '漢軍漢人', '安科爾沁', '貴州督標']

DATE (200개):
  ['甲寅', '三月', '十七年', '三十三年', '初八', '今春', '乙未', '四十二年', '宋朝', '元旦', '戊辰', '四十七年', '正月初一', '二十二日', '五十四年', '初四', '二年', '二十八年', '壬辰', '丙寅']

SYS (221개):
  ['結案', '三跪九叩', '太學', '科場', '流官', '徭役'

In [6]:
# Cell 5.2 (선택): 트랜잭션 크기 미리보기
print("=" * 60)
print("[월/연도별 트랜잭션 크기 예측]")
print("=" * 60)

# NONE/UNK 제외한 단어만
analyzable_words = set(ck[~ck['category'].isin(['NONE','UNK'])]['word'])
df_filt = merged[merged['word'].isin(analyzable_words)]

# 월별
month_size = df_filt.groupby('period_month')['word'].nunique()
print(f"\n월별 트랜잭션 ({len(month_size)}건):")
print(f"  평균 단어 수: {month_size.mean():.1f}")
print(f"  중앙값:      {month_size.median():.0f}")
print(f"  최소~최대:   {month_size.min()} ~ {month_size.max()}")

# 연도별
year_size = df_filt.groupby('period_year')['word'].nunique()
print(f"\n연도별 트랜잭션 ({len(year_size)}건):")
print(f"  평균 단어 수: {year_size.mean():.1f}")
print(f"  중앙값:      {year_size.median():.0f}")
print(f"  최소~최대:   {year_size.min()} ~ {year_size.max()}")


[월/연도별 트랜잭션 크기 예측]

월별 트랜잭션 (758건):
  평균 단어 수: 27.9
  중앙값:      25
  최소~최대:   4 ~ 124

연도별 트랜잭션 (62건):
  평균 단어 수: 240.3
  중앙값:      242
  최소~최대:   152 ~ 347


In [8]:
# Cell 5.5: 추가 수정 --- 아직 75%라 더 하고 살펴봐야 할 듯 ---
# 해보니까 굉장히 좋아서 하지 않고 진행하기로 함.

In [9]:
# Cell 6: 트랜잭션 CSV 생성
import pandas as pd

INPUT_NER  = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\ner_results.csv"
OUT_MONTH  = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\transactions_month.csv"
OUT_YEAR   = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\transactions_year.csv"

# ───── 옵션 ─────
MIN_FREQ        = 1       # 트랜잭션 크기 적정해서 필터링 불필요
USE_CATEGORIES  = None    # 전체 카테고리. 나중에 ["PER","LOC","OFF","GRP","EVT"] 등으로 좁혀볼 수 있음
# ─────────────────

df = pd.read_csv(INPUT_NER)
print(f"[INFO] NER 결과 로드: {len(df):,}행, 고유 단어 {df['word'].nunique():,}개")
print(f"[INFO] 카테고리 분포:\n{df['category'].value_counts().to_string()}\n")

if USE_CATEGORIES:
    before = len(df)
    df = df[df["category"].isin(USE_CATEGORIES)].reset_index(drop=True)
    print(f"[FILTER] 카테고리 {USE_CATEGORIES}: {before:,} → {len(df):,}행")

if MIN_FREQ > 1:
    before = len(df)
    df = df[df["freq"] >= MIN_FREQ].reset_index(drop=True)
    print(f"[FILTER] freq >= {MIN_FREQ}: {before:,} → {len(df):,}행")


def build_transactions(df, group_col, out_path):
    tx = (
        df.groupby(group_col)["word"]
        .apply(lambda x: sorted(set(x)))
        .reset_index()
    )
    tx["item_count"] = tx["word"].apply(len)
    tx["items"]      = tx["word"].apply(lambda lst: "|".join(lst))
    tx = tx[[group_col, "items", "item_count"]]
    tx.to_csv(out_path, index=False, encoding="utf-8-sig")

    print(f"\n[DONE] {group_col} 트랜잭션 → {out_path}")
    print(f"  트랜잭션 수: {len(tx):,}건")
    print(f"  아이템 수 — 평균: {tx['item_count'].mean():.1f}, "
          f"중앙값: {tx['item_count'].median():.0f}, "
          f"최소: {tx['item_count'].min()}, 최대: {tx['item_count'].max()}")
    return tx


month_tx = build_transactions(df, "period_month", OUT_MONTH)
year_tx  = build_transactions(df, "period_year",  OUT_YEAR)

print(f"\n[월별 샘플]")
print(month_tx.head(3)[["period_month", "item_count"]].to_string(index=False))
print(f"\n[연도별 샘플]")
print(year_tx.head(3)[["period_year", "item_count"]].to_string(index=False))


[INFO] NER 결과 로드: 21,126행, 고유 단어 4,903개
[INFO] 카테고리 분포:
category
OFF     5547
DATE    4761
PER     4406
LOC     4071
SYS     1173
GRP     1139
EVT       29


[DONE] period_month 트랜잭션 → C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\transactions_month.csv
  트랜잭션 수: 758건
  아이템 수 — 평균: 27.9, 중앙값: 25, 최소: 4, 최대: 124

[DONE] period_year 트랜잭션 → C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\transactions_year.csv
  트랜잭션 수: 62건
  아이템 수 — 평균: 240.3, 중앙값: 242, 최소: 152, 최대: 347

[월별 샘플]
period_month  item_count
     1661-01          29
     1661-02          34
     1661-03          60

[연도별 샘플]
 period_year  item_count
        1661         333
        1662         179
        1663         243


In [2]:
# Cell 7: FP-Growth 장바구니 분석 (월별 전용)
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

INPUT_MONTH = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\transactions_month.csv"
INPUT_NER   = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\ner_results.csv"
OUT_MONTH   = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\rules_month.csv"

# ───── 파라미터 ─────
MIN_SUPPORT    = 0.05    # 758개월 중 38개월 이상 등장
MIN_CONFIDENCE = 0.5
MIN_LIFT       = 1.5
MAX_LEN        = 3

# ───── 1. 트랜잭션 로드 ─────
df_m = pd.read_csv(INPUT_MONTH)
transactions = df_m["items"].apply(lambda s: s.split("|")).tolist()
print(f"[INFO] 월별 트랜잭션: {len(transactions):,}건")

# ───── 2. 인코딩 + FP-Growth ─────
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
te_df = pd.DataFrame(te_array, columns=te.columns_)
print(f"[INFO] 고유 아이템: {len(te.columns_):,}개")

freq_items = fpgrowth(te_df, min_support=MIN_SUPPORT, use_colnames=True, max_len=MAX_LEN)
print(f"[INFO] 빈발 아이템셋: {len(freq_items):,}개")

# ───── 3. 연관규칙 ─────
rules = association_rules(freq_items, metric="confidence", min_threshold=MIN_CONFIDENCE)
rules = rules[rules["lift"] >= MIN_LIFT].copy()

rules["antecedents"] = rules["antecedents"].apply(lambda x: ", ".join(sorted(x)))
rules["consequents"] = rules["consequents"].apply(lambda x: ", ".join(sorted(x)))
rules["ant_len"]    = rules["antecedents"].apply(lambda s: len(s.split(", ")))
rules["cons_len"]   = rules["consequents"].apply(lambda s: len(s.split(", ")))
rules = rules.sort_values(["lift","confidence"], ascending=False).reset_index(drop=True)
rules = rules[["antecedents","consequents","ant_len","cons_len","support","confidence","lift"]]

# ───── 4. 카테고리 태그 ─────
ner = pd.read_csv(INPUT_NER)
word2cat = dict(zip(ner["word"], ner["category"]))
def tag(s): return ", ".join(f"{w}({word2cat.get(w,'?')})" for w in s.split(", "))
rules["antecedents_tagged"] = rules["antecedents"].apply(tag)
rules["consequents_tagged"] = rules["consequents"].apply(tag)

# ───── 5. 저장 + 요약 ─────
rules.to_csv(OUT_MONTH, index=False, encoding="utf-8-sig")
print(f"\n[DONE] 월별 룰 {len(rules):,}개 → {OUT_MONTH}")
print(f"\n[상위 15개 (lift 기준)]")
print(rules.head(15)[["antecedents_tagged","consequents_tagged","support","confidence","lift"]].to_string(index=False))


[INFO] 월별 트랜잭션: 758건
[INFO] 고유 아이템: 4,903개
[INFO] 빈발 아이템셋: 673개

[DONE] 월별 룰 1,475개 → C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\rules_month.csv

[상위 15개 (lift 기준)]
antecedents_tagged consequents_tagged  support  confidence      lift
           固山(SYS)            貝子(OFF) 0.051451    0.928571 15.996753
           貝子(OFF)            固山(SYS) 0.051451    0.886364 15.996753
          兼禮部(OFF)   侍郎(OFF), 內閣(OFF) 0.050132    0.950000 15.321277
  侍郎(OFF), 內閣(OFF)           兼禮部(OFF) 0.050132    0.808511 15.321277
           藩王(OFF)   元旦(DATE), 王(OFF) 0.050132    0.678571 13.535714
  元旦(DATE), 王(OFF)            藩王(OFF) 0.050132    1.000000 13.535714
         朝鮮國王(PER)   堂子(LOC), 滿洲(GRP) 0.051451    0.709091 13.437273
  堂子(LOC), 滿洲(GRP)          朝鮮國王(PER) 0.051451    0.975000 13.437273
  八旗(SYS), 堂子(LOC)          朝鮮國王(PER) 0.052770    0.952381 13.125541
         朝鮮國王(PER)   八旗(SYS), 堂子(LOC) 0.052770    0.727273 13.125541
         朝鮮國王(PER)  堂子(LOC), 正月(DATE) 0.071240    0.981818 13.056459
      

In [4]:
# Cell 8: 룰 탐색
import pandas as pd

rules = pd.read_csv(r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\rules_month.csv")


def search_rules(keyword, top=20):
    mask = (rules["antecedents"].str.contains(keyword, na=False) |
            rules["consequents"].str.contains(keyword, na=False))
    result = rules[mask].head(top)
    print(f"\n[검색: '{keyword}'] {mask.sum()}개 / 상위 {len(result)}개")
    cols = ["antecedents_tagged","consequents_tagged","support","confidence","lift"]
    print(result[cols].to_string(index=False))


def filter_by_category(cat_pattern, top=20):
    mask = (rules["antecedents_tagged"].str.contains(cat_pattern, regex=True, na=False) |
            rules["consequents_tagged"].str.contains(cat_pattern, regex=True, na=False))
    result = rules[mask].head(top)
    print(f"\n[카테고리 패턴: {cat_pattern}] {mask.sum()}개 / 상위 {len(result)}개")
    cols = ["antecedents_tagged","consequents_tagged","support","confidence","lift"]
    print(result[cols].to_string(index=False))


# ───── 사용 예시 ─────
# search_rules("噶爾丹")            # 갈단(준가르) 관련
# search_rules("朝鮮")              # 조선 관련
# search_rules("三藩")              # 삼번 관련
# filter_by_category(r"\(EVT\)")    # 사건 포함 룰
# filter_by_category(r"\(PER\).*\(LOC\)")  # 인물+지명 룰

search_rules("朝鮮")              # 조선 관련

filter_by_category(r"\(EVT\)", top=20)



[검색: '朝鮮'] 273개 / 상위 20개
  antecedents_tagged consequents_tagged  support  confidence      lift
           朝鮮國王(PER)   堂子(LOC), 滿洲(GRP) 0.051451    0.709091 13.437273
    堂子(LOC), 滿洲(GRP)          朝鮮國王(PER) 0.051451    0.975000 13.437273
    八旗(SYS), 堂子(LOC)          朝鮮國王(PER) 0.052770    0.952381 13.125541
           朝鮮國王(PER)   八旗(SYS), 堂子(LOC) 0.052770    0.727273 13.125541
           朝鮮國王(PER)  堂子(LOC), 正月(DATE) 0.071240    0.981818 13.056459
           朝鮮國王(PER)  元旦(DATE), 堂子(LOC) 0.071240    0.981818 13.056459
           朝鮮國王(PER)   堂子(LOC), 朔(DATE) 0.071240    0.981818 13.056459
   堂子(LOC), 正月(DATE)          朝鮮國王(PER) 0.071240    0.947368 13.056459
   元旦(DATE), 堂子(LOC)          朝鮮國王(PER) 0.071240    0.947368 13.056459
    堂子(LOC), 朔(DATE)          朝鮮國王(PER) 0.071240    0.947368 13.056459
    堂子(LOC), 藩王(OFF)          朝鮮國王(PER) 0.063325    0.941176 12.971123
   堂子(LOC), 大學士(OFF)          朝鮮國王(PER) 0.063325    0.941176 12.971123
           朝鮮國王(PER)   堂子(LOC), 藩王(OFF) 0.063325   

In [5]:
# 진단: EVT 가 없었음
import pandas as pd
rules = pd.read_csv(r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\rules_month.csv")

# 1. tagged 컬럼 첫 5개 실제 형태 확인 (공백/특수문자 있는지)
print("[1] tagged 컬럼 실제 형태:")
for s in rules["antecedents_tagged"].head(5):
    print(f"  '{s}'  (길이 {len(s)})")

# 2. EVT 문자열이 어디든 들어있는지 단순 검색
mask_simple = (rules["antecedents_tagged"].str.contains("EVT", na=False) |
               rules["consequents_tagged"].str.contains("EVT", na=False))
print(f"\n[2] 'EVT' 단순 포함 룰: {mask_simple.sum()}개")

# 3. EVT 단어 목록
ner = pd.read_csv(r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\ner_results.csv")
evt_words = ner[ner["category"]=="EVT"]["word"].unique()
print(f"\n[3] EVT 단어 ({len(evt_words)}개): {list(evt_words)}")

# 4. EVT 단어가 월별 트랜잭션에 몇 번 등장했는지
tx = pd.read_csv(r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\transactions_month.csv")
print(f"\n[4] EVT 단어별 등장 월 수 (758개월 중):")
for w in evt_words:
    count = tx["items"].str.contains(w, na=False).sum()
    pct = count / len(tx) * 100
    print(f"  {w}: {count}개월 ({pct:.1f}%)  → support 임계값(5%, 38개월) {'통과' if pct>=5 else '미달'}")


[1] tagged 컬럼 실제 형태:
  '固山(SYS)'  (길이 7)
  '貝子(OFF)'  (길이 7)
  '兼禮部(OFF)'  (길이 8)
  '侍郎(OFF), 內閣(OFF)'  (길이 16)
  '藩王(OFF)'  (길이 7)

[2] 'EVT' 단순 포함 룰: 0개

[3] EVT 단어 (10개): ['南巡', '喀爾喀會盟', '靖難', '喀爾喀降', '平噶爾丹', '征噶爾丹', '耿逆之變', '三逆', '征拉藏', '平三逆']

[4] EVT 단어별 등장 월 수 (758개월 중):
  南巡: 19개월 (2.5%)  → support 임계값(5%, 38개월) 미달
  喀爾喀會盟: 2개월 (0.3%)  → support 임계값(5%, 38개월) 미달
  靖難: 1개월 (0.1%)  → support 임계값(5%, 38개월) 미달
  喀爾喀降: 1개월 (0.1%)  → support 임계값(5%, 38개월) 미달
  平噶爾丹: 1개월 (0.1%)  → support 임계값(5%, 38개월) 미달
  征噶爾丹: 1개월 (0.1%)  → support 임계값(5%, 38개월) 미달
  耿逆之變: 1개월 (0.1%)  → support 임계값(5%, 38개월) 미달
  三逆: 2개월 (0.3%)  → support 임계값(5%, 38개월) 미달
  征拉藏: 1개월 (0.1%)  → support 임계값(5%, 38개월) 미달
  平三逆: 1개월 (0.1%)  → support 임계값(5%, 38개월) 미달


In [ ]:
# 해결 처방전: EVT 전용 분석 (임계값 대폭 완화 --- 미달이었음 다)
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

INPUT_MONTH = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\transactions_month.csv"
INPUT_NER   = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\ner_results.csv"
OUT_EVT     = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\rules_month_evt.csv"

# EVT 단어 목록
ner = pd.read_csv(INPUT_NER)
evt_words = set(ner[ner["category"]=="EVT"]["word"])
print(f"[INFO] EVT 단어: {evt_words}")

# 트랜잭션 로드
df_m = pd.read_csv(INPUT_MONTH)
transactions = df_m["items"].apply(lambda s: s.split("|")).tolist()

# EVT 단어가 하나라도 들어간 월만 골라서 분석
evt_tx = [tx for tx in transactions if any(w in evt_words for w in tx)]
print(f"[INFO] EVT 등장 월: {len(evt_tx)}개월")

# 임계값을 매우 낮게
te = TransactionEncoder()
te_df = pd.DataFrame(te.fit(evt_tx).transform(evt_tx), columns=te.columns_)
freq = fpgrowth(te_df, min_support=0.15, use_colnames=True, max_len=3)
rules = association_rules(freq, metric="confidence", min_threshold=0.5)
rules = rules[rules["lift"] >= 2.0].copy()

# EVT가 포함된 룰만 필터링
def has_evt(itemset):
    return any(w in evt_words for w in itemset)
rules = rules[rules["antecedents"].apply(has_evt) | rules["consequents"].apply(has_evt)]

# 정리
rules["antecedents"] = rules["antecedents"].apply(lambda x: ", ".join(sorted(x)))
rules["consequents"] = rules["consequents"].apply(lambda x: ", ".join(sorted(x)))
rules = rules.sort_values("lift", ascending=False).reset_index(drop=True)

# 카테고리 태그
word2cat = dict(zip(ner["word"], ner["category"]))
def tag(s): return ", ".join(f"{w}({word2cat.get(w,'?')})" for w in s.split(", "))
rules["antecedents_tagged"] = rules["antecedents"].apply(tag)
rules["consequents_tagged"] = rules["consequents"].apply(tag)

print(f"\n[DONE] EVT 룰: {len(rules)}개")
print(rules.head(20)[["antecedents_tagged","consequents_tagged","support","confidence","lift"]].to_string(index=False))

rules.to_csv(OUT_EVT, index=False, encoding="utf-8-sig")


In [6]:
!pip install python-dotenv



[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: C:\Users\USER\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
